---

In [1]:
import stim
from fractions import Fraction

In [2]:
qasm = stim.Tableau.from_stabilizers([stim.PauliString("XX"),stim.PauliString("ZZ"),]).to_circuit().to_qasm(open_qasm_version=3)
qasm2 = stim.Tableau.from_stabilizers([stim.PauliString("IIIXXXX"),stim.PauliString("IXXIIXX"),stim.PauliString("XIXIXIX"), stim.PauliString("IIIZZZZ"), stim.PauliString("IZZIIZZ"), stim.PauliString("ZIZIZIZ")], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3)
qasm_5qubit = stim.Tableau.from_stabilizers([
    stim.PauliString("XZZXI"),
    stim.PauliString("IXZZX"), 
    stim.PauliString("XIXZZ"),
    stim.PauliString("ZXIXZ")
], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3)
shor_code = stim.Tableau.from_stabilizers([
    stim.PauliString("ZZIIIIIII"),
    stim.PauliString("ZIZIIIIII"),
    stim.PauliString("IIIZZIIII"),
    stim.PauliString("IIIZIZIII"),
    stim.PauliString("IIIIIIZZI"),
    stim.PauliString("IIIIIIZIZ"),
    stim.PauliString("XXXXXXIII"),
    stim.PauliString("IIIXXXXXX"),
], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3)
qasm_7qubit = stim.Tableau.from_stabilizers([
    stim.PauliString("IIIXXXX"),
    stim.PauliString("IXXIIXX"),
    stim.PauliString("XIXIXIX"),
    stim.PauliString("IIIZZZZ"),
    stim.PauliString("IZZIIZZ"),
    stim.PauliString("ZIZIZIZ")
], allow_underconstrained=True).to_circuit(method="graph_state").to_qasm(open_qasm_version=3) # O(n^2)
# shor_code.diagram()

# s = stim.TableauSimulator()
# s.do_circuit(shor_code)
# s.canonical_stabilizers()
qasm_7qubit


'OPENQASM 3.0;\ninclude "stdgates.inc";\ndef rx(qubit q0) { reset q0; h q0; }\n\nqreg q[7];\n\nrx(q[0]);\nrx(q[1]);\nrx(q[2]);\nrx(q[3]);\nrx(q[4]);\nrx(q[5]);\nrx(q[6]);\nbarrier q;\n\ncz q[0], q[1];\ncz q[0], q[2];\ncz q[1], q[4];\ncz q[1], q[5];\ncz q[2], q[4];\ncz q[2], q[6];\ncz q[3], q[4];\ncz q[3], q[5];\ncz q[3], q[6];\nbarrier q;\n\nh q[0];\nh q[4];\nh q[5];\nh q[6];\n'

In [ ]:
# from ..pyzx.graph_states import GraphState
from pyzx import Circuit
from pyzx import *
from pyzx import to_graph_like, clifford_simp
from pyzx import draw_d3
from pyzx import GraphState
import stim
import re

def stim_qasm3_to_pyzx_qasm2(qasm: str, decompose_cz: bool = False) -> str:
    q = qasm
    q = re.sub(r'OPENQASM 3\.0;', 'OPENQASM 2.0;', q)
    q = re.sub(r'include "stdgates\.inc";\n', 'include "qelib1.inc";\n', q)
    # remove rx definition
    q = re.sub(r'def\s+rx\(qubit q0\)\s*\{[^}]*\}\n+', '', q)
    # replace rx(q[i]); with h q[i];
    q = re.sub(r'rx\s*\(\s*q\[(\d+)\]\s*\)\s*;', r'h q[\1];', q)
    # remove reset if still present
    q = re.sub(r'reset\s+q\[(\d+)\];', '', q)
    # remove barriers
    q = re.sub(r'barrier q;\n', '', q)
    # compress blank lines
    q = re.sub(r'\n\s*\n+', '\n', q).strip() + '\n'
    if decompose_cz:
        lines = []
        for line in q.splitlines():
            m = re.match(r'cz q\[(\d+)\],\s*q\[(\d+)\];', line.strip())
            if m:
                a, b = m.groups()
                lines.append(f'h q[{b}];')
                lines.append(f'cx q[{a}], q[{b}];')
                lines.append(f'h q[{b}];')
            else:
                lines.append(line)
        q = '\n'.join(lines) + '\n'
    return q

def tableau_to_graph(list, quiet = True):
    n = len(list[0])
    k = n - len(list)
    stabilizers = [stim.PauliString(x) for x in list]
    tableau = stim.Tableau.from_stabilizers(stabilizers, allow_underconstrained=True)
    stim_circ = tableau.to_circuit(method="elimination")
    qasm = stim_circ.to_qasm()
    pyzx_circ = Circuit.from_qasm(qasm)
    g = GraphState.from_circuit(pyzx_circ, k)
    g.to_canonical_form(quiet = quiet)
    return g
    
quiet = True
qasm_7qubit = stim_qasm3_to_pyzx_qasm2(qasm_7qubit, decompose_cz=False)
circ = Circuit.from_qasm(qasm_7qubit)

g = GraphState.from_circuit(circ, 1)    

g.to_canonical_form()

spider_simp: 10. 6. 3. 3. 1. 1.  6 iterations
id_simp: 4.  1 iterations
(6,)


In [11]:
import networkx as nx
from pyvis.network import Network
import streamlit as st

def pyzx_graph_to_pyvis(g):
    nxg = nx.Graph()
    for v in g.get_states():
        v_type = g.type(v)
        if g.get_bound(v) in g.inputs():
            color = "green"
        else:
            color = "blue"
        if v_type != VertexType.BOUNDARY:
            nxg.add_node(v, 
                        label=str(v), 
                        color=color, 
                        title=f"Type: {v_type.name}")
            
    for e in g.edges():
        edge_type = g.edge_type(g.edge(*e))
        edge_color = "red" if edge_type == EdgeType.HADAMARD else "black"
        if g.type(e[0]) != VertexType.BOUNDARY and g.type(e[1]) != VertexType.BOUNDARY:
            nxg.add_edge(e[0], e[1])
            nxg[e[0]][e[1]]['color'] = edge_color

    # # Create PyVis network
    net = Network(notebook=False, directed=False)
    net.from_nx(nxg)
    net.toggle_physics(True)
    return net

    # net.show_buttons(filter_=['physics'])

    # html_path = "pyzx_graph.html"
    # net.save_graph(html_path)  # ✅ does not require render()

st.title("Tableau")

2025-08-10 12:30:07.409 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-10 12:30:07.485 
  command:

    streamlit run /home/francesco/.local/lib/python3.10/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-08-10 12:30:07.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-10 12:30:07.487 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()